# Polarized Neutron Reflectometry — Magnetic Models and Simultaneous Multi-Channel Fitting

This notebook demonstrates the polarized (PNR) functionality of `easyreflectometry`:

| Feature | API |
|---|---|
| Magnetic layers as first-class model parameters | `LayerMagnetism(rho_m, theta_m)` on `Layer` |
| All four spin channels in one calculation | `interface.polarized_reflectivity_profiles(q, model_id)` |
| Nuclear + magnetic SLD profile | `interface.magnetic_sld_profile(model_id)` |
| Per-file spin-channel detection | `detect_polarization_channel(path)` |
| One experiment = one dataset per channel | `PolarizedDataSet` |
| Simultaneous fit of all measured channels | `MultiFitter.fit_polarized(data)` |

The magnetic parameters `rho_m` (magnetic SLD, in 10⁻⁶ Å⁻²) and `theta_m` (in-plane moment
angle, degrees) are ordinary `easyscience` `Parameter`s: they can be fixed or freed, bounded,
serialized, and fitted: together with the structural parameters, against **all measured spin
channels at once**. The refl1d backend computes the four spin cross-sections in a single kernel
evaluation and caches them per iteration, so an N-channel fit costs about as much as a
single-channel one.

**Convention** (refl1d): with the default guide field (`Aguide = 270°`), a moment at
`theta_m = 270°` is *aligned* with the field: the non-spin-flip channels see
ρ ± ρ_M and the spin-flip channels vanish. Any other angle cants the moment and
produces spin-flip scattering. Polarized calculations require the **refl1d** calculator.

## 1. Imports

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from easyreflectometry.calculators import CalculatorFactory
from easyreflectometry.data import PolarizedDataSet
from easyreflectometry.data import detect_polarization_channel
from easyreflectometry.data import load_as_dataset
from easyreflectometry.fitting import MultiFitter
from easyreflectometry.model import Model
from easyreflectometry.model import PercentageFwhm
from easyreflectometry.sample import Layer
from easyreflectometry.sample import LayerMagnetism
from easyreflectometry.sample import Material
from easyreflectometry.sample import Multilayer
from easyreflectometry.sample import Sample

%matplotlib inline

rng = np.random.default_rng(42)

CHANNEL_LABELS = {'pp': 'R++ (up-up)', 'pm': 'R+- (up-down)', 'mp': 'R-+ (down-up)', 'mm': 'R-- (down-down)'}
CHANNEL_COLORS = {'pp': 'C0', 'pm': 'C2', 'mp': 'C3', 'mm': 'C1'}

## 2. Build a Magnetic Sample

A single ferromagnetic iron film on silicon, measured in vacuum:

- **Vacuum** superphase
- **Fe film**, 200 Å: nuclear SLD 8.02·10⁻⁶ Å⁻², magnetic SLD `rho_m = 5.0`·10⁻⁶ Å⁻²
  (bulk Fe), moment canted at `theta_m = 40°` so that all four channels are non-trivial
- **Si** substrate

Attaching a `LayerMagnetism` to a layer is all that is needed: when the model is given a
calculator interface, magnetism is switched on automatically (and removing the last magnetic
layer switches it off again).

In [ ]:
TRUTH = {'thickness': 200.0, 'rho_m': 5.0, 'theta_m': 40.0}


def build_model(thickness=TRUTH['thickness'], rho_m=TRUTH['rho_m'], theta_m=TRUTH['theta_m'], name='PNR Model'):
    """Vacuum | Fe film (magnetic) | Si substrate, with a fresh refl1d calculator."""
    vacuum = Material(0.0, 0.0, 'Vacuum')
    iron = Material(8.02, 0.0, 'Fe')
    silicon = Material(2.07, 0.0, 'Si')

    superphase = Layer(vacuum, 0, 0, 'Vacuum superphase')
    film = Layer(
        iron, thickness, 5, 'Fe film',
        magnetism=LayerMagnetism(rho_m=rho_m, theta_m=theta_m, name='Fe moment'),
    )
    substrate = Layer(silicon, 0, 3, 'Si substrate')

    sample = Sample(Multilayer(superphase), Multilayer(film), Multilayer(substrate), name='Fe on Si')
    model = Model(sample, 1.0, 0.0, PercentageFwhm(2.0), name)

    interface = CalculatorFactory()
    interface.switch('refl1d')  # magnetism requires the refl1d backend
    model.interface = interface
    return model


truth_model = build_model()
print(truth_model)
print(f'has_magnetism            : {truth_model.has_magnetism}')
print(f'calculator magnetism flag: {truth_model.interface().include_magnetism}  (enabled automatically)')

## 3. Simulate All Four Spin Channels

`polarized_reflectivity_profiles` returns a dictionary keyed `'pp'`, `'pm'`, `'mp'`, `'mm'`.
All four cross-sections come from one refl1d kernel evaluation.

With the moment canted at 40° the non-spin-flip channels split (they see the moment's
projection on the field) and the spin-flip channels pick up the perpendicular component.
For a non-chiral, non-absorptive sample `pm` and `mp` coincide by symmetry.

In [ ]:
q = np.linspace(0.008, 0.22, 160)
channels_truth = truth_model.interface.polarized_reflectivity_profiles(q, truth_model.unique_name)

fig, ax = plt.subplots(figsize=(10, 6))
for channel, reflectivity in channels_truth.items():
    ax.plot(q, reflectivity, color=CHANNEL_COLORS[channel], lw=1.5, label=CHANNEL_LABELS[channel])
ax.set_yscale('log')
ax.set_xlabel('Q (Å⁻¹)')
ax.set_ylabel('Reflectivity')
ax.set_title('Simulated spin channels — Fe film, moment canted 40°')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Nuclear and Magnetic SLD Profile

`magnetic_sld_profile` returns `z`, nuclear ρ(z), magnetic ρ_M(z) and the moment angle θ_M(z).
The most intuitive PNR view adds the **spin-dependent potentials**: what each neutron spin
state actually "sees":

$$\rho_\pm(z) = \rho(z) \pm \rho_M(z)\,\cos\bigl(\theta_M(z) - A_\text{guide}\bigr)$$

In [ ]:
z, sld, rho_m_profile, theta_m_profile = truth_model.interface.magnetic_sld_profile(truth_model.unique_name)

AGUIDE = 270.0  # refl1d default guide-field angle
projection = rho_m_profile * np.cos(np.radians(theta_m_profile - AGUIDE))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(z, sld, 'k-', lw=2, label='nuclear ρ(z)')
ax.plot(z, rho_m_profile, 'C4-', lw=2, label='magnetic ρ$_M$(z)')
ax.plot(z, sld + projection, 'C0--', lw=1.5, label='spin-up potential ρ + ρ$_M$cos(θ$_M$−A)')
ax.plot(z, sld - projection, 'C1--', lw=1.5, label='spin-down potential ρ − ρ$_M$cos(θ$_M$−A)')
ax.set_xlabel('z (Å)')
ax.set_ylabel('SLD (10⁻⁶ Å⁻²)')
ax.set_title('Nuclear and magnetic SLD profile')
ax.legend(loc='upper right', fontsize=9)

ax2 = ax.twinx()
ax2.plot(z, theta_m_profile, 'C7:', lw=1.5)
ax2.set_ylabel('θ$_M$ (deg)', color='C7')
ax2.tick_params(axis='y', colors='C7')
plt.tight_layout()
plt.show()

## 5. Spin Asymmetry

The spin asymmetry

$$SA = \frac{R^{++} - R^{--}}{R^{++} + R^{--}}$$

removes most of the structural (nuclear) contribution and is visually far more sensitive to
weak magnetism than the raw reflectivities: the standard first look at any PNR measurement.

In [ ]:
spin_asymmetry = (channels_truth['pp'] - channels_truth['mm']) / (channels_truth['pp'] + channels_truth['mm'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(q, spin_asymmetry, 'C5-', lw=1.5)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Q (Å⁻¹)')
ax.set_ylabel('(R⁺⁺ − R⁻⁻) / (R⁺⁺ + R⁻⁻)')
ax.set_title('Spin asymmetry')
plt.tight_layout()
plt.show()

## 6. A Synthetic Experiment: One File per Channel

Polarized measurements typically arrive as **one file per spin channel**. We simulate that:
3% relative noise on each channel, written to four separate files whose names carry the
conventional channel suffixes (`_uu`, `_dd`, `_ud`, `_du`).

`detect_polarization_channel` identifies the channel of each file: from the ORSO header
(`instrument_settings.polarization`) when present, otherwise from filename tokens. Only the
four fully-analysed cross-sections `pp`/`pm`/`mp`/`mm` are ever assigned; partially-analysed
observables (`po`, `mo`, …) measure *channel sums* and are left for the user to decide.
(For GUI workflows, `Project.suggest_polarized_channel_assignment(paths)` wraps this per-file
and `Project.load_polarized_experiment({channel: path})` performs the load.)

In [ ]:
DATA_DIR = 'polarized_demo_data'
os.makedirs(DATA_DIR, exist_ok=True)

FILE_SUFFIX = {'pp': 'uu', 'pm': 'ud', 'mp': 'du', 'mm': 'dd'}
NOISE = 0.03

file_paths = []
for channel, reflectivity in channels_truth.items():
    sigma = NOISE * reflectivity
    noisy = np.clip(reflectivity + sigma * rng.standard_normal(len(q)), 1e-12, None)
    path = os.path.join(DATA_DIR, f'fe_on_si_{FILE_SUFFIX[channel]}.dat')
    np.savetxt(path, np.column_stack([q, noisy, sigma]), header='Qz (1/angstrom)  R  sR')
    file_paths.append(path)

print('Automatic channel detection:')
for path in file_paths:
    detected = detect_polarization_channel(path)
    print(f'  {os.path.basename(path):24s} -> {detected.value if detected else "(user must assign)"}')

## 7. Group the Channels into a `PolarizedDataSet`

A `PolarizedDataSet` holds one `DataSet1D` per measured channel (any subset of the four -
an NSF-only experiment would just have `pp` and `mm`) and one **shared model**. Channels are
kept in canonical order (pp, pm, mp, mm); the `channels` mapping is read-only, with validated
`set_channel` / `remove_channel` methods for editing.

> **Note** — `DataSet1D.ye` stores **variances** (σ²), following the scipp convention. The
> text-file loader squares the error column for you.

We start the fit model from deliberately wrong values: thickness 180 Å (truth 200),
`rho_m` 3.0 (truth 5.0), `theta_m` 60° (truth 40°).

In [ ]:
fit_model = build_model(thickness=180.0, rho_m=3.0, theta_m=60.0, name='PNR Fit Model')

data = PolarizedDataSet(
    name='Fe on Si (synthetic PNR)',
    channels={detect_polarization_channel(path): load_as_dataset(path) for path in file_paths},
    model=fit_model,
)
print(data)
print(f'channels: {[channel.value for channel in data.available_channels]}')
print(f"points per channel: {len(data['pp'].x)}")

## 8. Simultaneous Multi-Channel Fit

`MultiFitter.fit_polarized(data)` fits **all measured channels at once** against the one
shared model:

- structural parameters (thickness, roughness, nuclear SLD, scale, background) are common to
  every channel automatically;
- the magnetic parameters shape the channels through the spin-dependent kernel: the
  non-spin-flip splitting pins `rho_m·cos θ_m` while the spin-flip channels pin the
  perpendicular component, so `rho_m` and `theta_m` are individually well-determined;
- each iteration costs a single refl1d kernel evaluation thanks to the four-channel cache.

It returns one `FitResults` per channel.

In [ ]:
film_layer = fit_model.sample[1].layers[0]

film_layer.thickness.fixed = False
film_layer.thickness.min = 150
film_layer.thickness.max = 250
film_layer.magnetism.rho_m.fixed = False
film_layer.magnetism.rho_m.min = 0
film_layer.magnetism.rho_m.max = 8
film_layer.magnetism.theta_m.fixed = False
film_layer.magnetism.theta_m.min = 0
film_layer.magnetism.theta_m.max = 90

print('Free parameters (start values):')
for parameter in fit_model.get_fit_parameters():
    print(f'  {parameter.name:12s} = {float(parameter.value):8.3f}   bounds=[{parameter.min:g}, {parameter.max:g}]')

fitter = MultiFitter(fit_model)
results = fitter.fit_polarized(data)

print(f'\nsuccess: {all(result.success for result in results.values())}')
print(f'reduced chi² (all channels): {fitter.reduced_chi:.3f}')

print(f'\n{"Parameter":12s} {"truth":>10s} {"start":>10s} {"fitted":>10s}')
print('-' * 46)
starts = {'thickness': 180.0, 'rho_m': 3.0, 'theta_m': 60.0}
fitted = {
    'thickness': float(film_layer.thickness.value),
    'rho_m': float(film_layer.magnetism.rho_m.value),
    'theta_m': float(film_layer.magnetism.theta_m.value),
}
for key in TRUTH:
    print(f'{key:12s} {TRUTH[key]:>10.3f} {starts[key]:>10.3f} {fitted[key]:>10.3f}')

## 9. Fitted Curves per Channel

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

for ax, channel in zip(axes.flat, data.available_channels):
    dataset = data[channel]
    fitted_curve = fit_model.interface.reflectivity_profile_channel(dataset.x, fit_model.unique_name, channel)
    ax.errorbar(
        dataset.x, dataset.y, yerr=np.sqrt(dataset.ye),  # ye holds variances
        fmt='o', ms=2.5, alpha=0.45, color=CHANNEL_COLORS[channel.value], label='synthetic data',
    )
    ax.plot(dataset.x, fitted_curve, 'k-', lw=1.5, label='fit')
    ax.set_yscale('log')
    ax.set_title(CHANNEL_LABELS[channel.value])
    ax.legend(fontsize=9)

for ax in axes[1]:
    ax.set_xlabel('Q (Å⁻¹)')
for ax in axes[:, 0]:
    ax.set_ylabel('Reflectivity')

fig.suptitle('Simultaneous four-channel fit — all channels share one model', y=1.0)
plt.tight_layout()
plt.show()

### Spin asymmetry: data vs fit

In [ ]:
fitted_channels = fit_model.interface.polarized_reflectivity_profiles(q, fit_model.unique_name)

sa_data = (data['pp'].y - data['mm'].y) / (data['pp'].y + data['mm'].y)
sa_fit = (fitted_channels['pp'] - fitted_channels['mm']) / (fitted_channels['pp'] + fitted_channels['mm'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(q, sa_data, 'o', ms=3, alpha=0.5, color='C5', label='synthetic data')
ax.plot(q, sa_fit, 'k-', lw=1.5, label='fit')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Q (Å⁻¹)')
ax.set_ylabel('(R⁺⁺ − R⁻⁻) / (R⁺⁺ + R⁻⁻)')
ax.set_title('Spin asymmetry — data vs simultaneous fit')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

- **`LayerMagnetism(rho_m, theta_m)`** makes a layer magnetic; both are fittable, bounded,
  serializable `Parameter`s. Magnetism is enabled on the calculator automatically (refl1d
  only).
- **`polarized_reflectivity_profiles`** / **`magnetic_sld_profile`** give the four spin
  channels and the nuclear + magnetic depth profile in one call each.
- **`detect_polarization_channel`** assigns spin channels from ORSO headers or filename
  tokens; partially-analysed observables (`po`/`mo` = channel sums) are never auto-assigned.
- **`PolarizedDataSet`** groups per-channel datasets (2-channel NSF-only works the same way:
  provide just `pp` and `mm`) under one shared model; `ye` holds variances.
- **`MultiFitter.fit_polarized`** fits every measured channel simultaneously and recovered
  thickness, `rho_m` and `theta_m` here to within a fraction of a percent, at roughly the
  cost of a single-channel fit (four-channel cache: one kernel evaluation per iteration).

For file-based / GUI workflows the same functionality is reachable through
`Project.suggest_polarized_channel_assignment(paths)` and
`Project.load_polarized_experiment({channel: path})`.